In [ ]:
!pip install pandas scikit-learn matplotlib seaborn plotly tensorflow

In [ ]:
# Importa??o das bibliotecas necess?rias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Nome das colunas com base em adult.names
column_names = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]

# Carregamento dos Dados
base_censo = pd.read_csv('data/adult.data', names=column_names, sep=", ", engine='python')
print("Distribui??o das classes antes do balanceamento:")
print(base_censo['income'].value_counts())



In [ ]:
# Separa??o de Caracter?sticas - Features (X) e Alvo (Target - y)
X_censo = base_censo.iloc[:, 0:14].values
y_censo = base_censo.iloc[:, 14].values

# Codifica??o das Vari?veis Categ?ricas com Label Encoding
label_encoder_workclass = LabelEncoder()
label_encoder_education = LabelEncoder()
label_encoder_marital = LabelEncoder()
label_encoder_occupation = LabelEncoder()
label_encoder_relationship = LabelEncoder()
label_encoder_race = LabelEncoder()
label_encoder_sex = LabelEncoder()
label_encoder_country = LabelEncoder()

X_censo[:,1] = label_encoder_workclass.fit_transform(X_censo[:,1])
X_censo[:,3] = label_encoder_education.fit_transform(X_censo[:,3])
X_censo[:,5] = label_encoder_marital.fit_transform(X_censo[:,5])
X_censo[:,6] = label_encoder_occupation.fit_transform(X_censo[:,6])
X_censo[:,7] = label_encoder_relationship.fit_transform(X_censo[:,7])
X_censo[:,8] = label_encoder_race.fit_transform(X_censo[:,8])
X_censo[:,9] = label_encoder_sex.fit_transform(X_censo[:,9])
X_censo[:,13] = label_encoder_country.fit_transform(X_censo[:,13])




In [ ]:
# Codifica??o com OneHot Encoding
onehotencoder_censo = ColumnTransformer(
    transformers=[('OneHot', OneHotEncoder(), [1,3,5,6,7,8,9,13])],
    remainder='passthrough'
)
X_censo = onehotencoder_censo.fit_transform(X_censo).toarray()



In [ ]:
# Balanceamento dos Dados
X_50k = X_censo[y_censo == '<=50K']
y_50k = y_censo[y_censo == '<=50K']
X_mais_50k = X_censo[y_censo == '>50K']
y_mais_50k = y_censo[y_censo == '>50K']

# Amostragem de treinamento com 15% para <=50K e 40% para >50K
X_treinamento_50k, y_treinamento_50k = resample(X_50k, y_50k, replace=False, n_samples=max(int(0.15 * len(y_50k)), 1), random_state=0)
X_treinamento_mais_50k, y_treinamento_mais_50k = resample(X_mais_50k, y_mais_50k, replace=False, n_samples=max(int(0.40 * len(y_mais_50k)), 1), random_state=0)

# Conjunto de teste com o restante
X_teste_50k, y_teste_50k = resample(X_50k, y_50k, replace=False, n_samples=len(y_50k) - len(y_treinamento_50k), random_state=0)
X_teste_mais_50k, y_teste_mais_50k = resample(X_mais_50k, y_mais_50k, replace=False, n_samples=len(y_mais_50k) - len(y_treinamento_mais_50k), random_state=0)

# Combinar conjuntos balanceados
X_censo_treinamento = np.vstack((X_treinamento_50k, X_treinamento_mais_50k))
y_censo_treinamento = np.hstack((y_treinamento_50k, y_treinamento_mais_50k))
X_teste_final = np.vstack((X_teste_50k, X_teste_mais_50k))
y_teste_final = np.hstack((y_teste_50k, y_teste_mais_50k))

# Convertendo a vari?vel alvo para bin?ria
y_censo_treinamento = np.where(y_censo_treinamento == '>50K', 1, 0)
y_teste_final = np.where(y_teste_final == '>50K', 1, 0)



In [ ]:
# Normaliza??o dos Dados
scaler = StandardScaler()
X_censo_treinamento = scaler.fit_transform(X_censo_treinamento)
X_teste_final = scaler.transform(X_teste_final)

In [ ]:
# Constru??o do Modelo de Rede Neural com TensorFlow
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_censo_treinamento.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') 
])

# Rodar o Modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])



In [ ]:
# Treinamento do Modelo
history = model.fit(X_censo_treinamento, y_censo_treinamento, epochs=200, batch_size=32, validation_split=0.2)

# Avalia??o do Modelo
y_pred_proba = model.predict(X_teste_final)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
acuracia_final = accuracy_score(y_teste_final, y_pred)
print("\nAcur?cia do modelo com o conjunto de teste ajustado:", acuracia_final)



In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label='Acur?cia de Treinamento')
plt.plot(history.history['val_accuracy'], label='Acur?cia de Valida??o')
plt.title('Acur?cia do Modelo durante o Treinamento')
plt.xlabel('?pocas')
plt.ylabel('Acur?cia')
plt.legend()
plt.show()

In [ ]:
# Exibi??o da Matriz de Confus?o
matriz_confusao_final = confusion_matrix(y_teste_final, y_pred)
print("\nMatriz de Confus?o:")
print(matriz_confusao_final)

# Exibindo a matriz de confus?o de forma gr?fica
disp_final = ConfusionMatrixDisplay(confusion_matrix=matriz_confusao_final, display_labels=['<=50K', '>50K'])
disp_final.plot(cmap="Blues")
plt.show()

# Relat?rio de Classifica??o com foco na classe >50K
relatorio_classificacao_final = classification_report(y_teste_final, y_pred, target_names=['<=50K', '>50K'])
print("\nRelat?rio de Classifica??o:")
print(relatorio_classificacao_final)

In [ ]:
# MLClassifier

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
base_censo = pd.read_csv('data/adult.data', names=column_names, sep=", ", engine='python')
print("Distribui??o das classes antes do balanceamento:")
print(base_censo['income'].value_counts())

In [ ]:
X_censo = base_censo.iloc[:, 0:14].values
y_censo = base_censo.iloc[:, 14].values

In [ ]:
label_encoder_workclass = LabelEncoder()
label_encoder_education = LabelEncoder()
label_encoder_marital = LabelEncoder()
label_encoder_occupation = LabelEncoder()
label_encoder_relationship = LabelEncoder()
label_encoder_race = LabelEncoder()
label_encoder_sex = LabelEncoder()
label_encoder_country = LabelEncoder()

X_censo[:,1] = label_encoder_workclass.fit_transform(X_censo[:,1])
X_censo[:,3] = label_encoder_education.fit_transform(X_censo[:,3])
X_censo[:,5] = label_encoder_marital.fit_transform(X_censo[:,5])
X_censo[:,6] = label_encoder_occupation.fit_transform(X_censo[:,6])
X_censo[:,7] = label_encoder_relationship.fit_transform(X_censo[:,7])
X_censo[:,8] = label_encoder_race.fit_transform(X_censo[:,8])
X_censo[:,9] = label_encoder_sex.fit_transform(X_censo[:,9])
X_censo[:,13] = label_encoder_country.fit_transform(X_censo[:,13])

In [ ]:
onehotencoder_censo = ColumnTransformer(
    transformers=[('OneHot', OneHotEncoder(), [1,3,5,6,7,8,9,13])],
    remainder='passthrough'
)
X_censo = onehotencoder_censo.fit_transform(X_censo).toarray()

In [ ]:
X_50k = X_censo[y_censo == '<=50K']
y_50k = y_censo[y_censo == '<=50K']
X_mais_50k = X_censo[y_censo == '>50K']
y_mais_50k = y_censo[y_censo == '>50K']

# Garantir que o tamanho da amostra seja ao menos 1
sample_size_50k = max(min(int(0.30 * len(y_50k)), len(y_50k)), 1)
sample_size_mais_50k = max(min(int(0.30 * len(y_mais_50k)), len(y_mais_50k)), 1)

# Resampling com tamanho ajustado
X_treinamento_50k, y_treinamento_50k = resample(X_50k, y_50k, replace=False, n_samples=sample_size_50k, random_state=0)
X_treinamento_mais_50k, y_treinamento_mais_50k = resample(X_mais_50k, y_mais_50k, replace=False, n_samples=sample_size_mais_50k, random_state=0)

# Conjunto de teste balanceado
X_teste_50k, y_teste_50k = resample(X_50k, y_50k, replace=False, n_samples=max(5500, 1), random_state=0)
X_teste_mais_50k, y_teste_mais_50k = resample(X_mais_50k, y_mais_50k, replace=False, n_samples=max(2500, 1), random_state=0)

# Combinar conjuntos balanceados
X_censo_treinamento = np.vstack((X_treinamento_50k, X_treinamento_mais_50k))
y_censo_treinamento = np.hstack((y_treinamento_50k, y_treinamento_mais_50k))
X_teste_final = np.vstack((X_teste_50k, X_teste_mais_50k))
y_teste_final = np.hstack((y_teste_50k, y_teste_mais_50k))

In [ ]:
sample_size_50k

In [ ]:
sample_size_mais_50k

In [ ]:
#Normaliza??o dos dados
scaler = StandardScaler()
X_censo_treinamento = scaler.fit_transform(X_censo_treinamento)
X_teste_final = scaler.transform(X_teste_final)

In [ ]:
# Configura??o do Modelo de Rede Neural
mlp_classifier = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)


In [ ]:
# Treinamento do Modelo
mlp_classifier.fit(X_censo_treinamento, y_censo_treinamento)

In [ ]:
# Avalia??o do Modelo
previsoes_final = mlp_classifier.predict(X_teste_final)
acuracia_final = accuracy_score(y_teste_final, previsoes_final)
print("Acur?cia do modelo MLP com o conjunto de teste ajustado:", acuracia_final)


In [ ]:
# Exibi??o da Matriz de Confus?o
matriz_confusao_final = confusion_matrix(y_teste_final, previsoes_final)
print("Matriz de Confus?o:")
print(matriz_confusao_final)

In [ ]:
disp_final = ConfusionMatrixDisplay(confusion_matrix=matriz_confusao_final, display_labels=mlp_classifier.classes_)
disp_final.plot(cmap="Blues")
plt.show()

In [ ]:
#Relat?rio de Classifica??o
relatorio_classificacao_final = classification_report(y_teste_final, previsoes_final, target_names=['<=50K', '>50K'])
print("Relat?rio de Classifica??o:")
print(relatorio_classificacao_final)